# Weighted Tikhonov inverse problem on non-Euclidean spaces

This notebook is a compact tutorial for the SpaceCore weighted Tikhonov worked example.
It is not a generic Euclidean least-squares example. The point is that the spaces carry
inner products, and those inner products determine the adjoint.

Central lesson: **the coordinate matrix `M` does not determine the adjoint by itself.**
The adjoint is determined by `M` together with the inner products on the domain `X` and
codomain `Y`. SpaceCore's `A.H` / `rapply` represents this metric adjoint, not blindly
the coordinate transpose.

## Mathematical problem

Let

$$
X = \mathbb{R}^n, \qquad Y = \mathbb{R}^m.
$$

The spaces are not equipped with Euclidean coordinate inner products. Instead,

$$
\langle x, z\rangle_X = x^T G_X z, \qquad
\langle y, w\rangle_Y = y^T G_Y w,
$$

where

$$
G_X \in \mathbb{R}^{n \times n}, \qquad
G_Y \in \mathbb{R}^{m \times m}
$$

are symmetric positive-definite metric matrices. The forward operator is

$$
A : X \to Y
$$

with coordinate matrix

$$
M \in \mathbb{R}^{m \times n}.
$$

Given data $b \in Y$ and $\lambda > 0$, solve

$$
J(x) = \frac{1}{2}(Mx - b)^T G_Y (Mx - b)
     + \frac{\lambda}{2} x^T G_X x.
$$

The metric adjoint $A^\sharp : Y \to X$ is defined by

$$
\langle Ax, y\rangle_Y = \langle x, A^\sharp y\rangle_X.
$$

Substituting the weighted inner products gives

$$
(Mx)^T G_Y y = x^T G_X A^\sharp y.
$$

Since this must hold for every $x$, the coordinate formula is

$$
A^\sharp = G_X^{-1} M^T G_Y.
$$

The first-order condition is therefore

$$
(A^\sharp A + \lambda I_X)x = A^\sharp b.
$$

Multiplying by $G_X$ gives the dense NumPy reference system

$$
(M^T G_Y M + \lambda G_X)x = M^T G_Y b.
$$

In [1]:
from pathlib import Path
import sys

import numpy as np
import spacecore as sc

# Make the source checkout importable whether the notebook is run from the
# repository root or from inside the tutorials/ directory.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "examples" / "weighted_tikhonov.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find the SpaceCore repository root.")

from examples.weighted_tikhonov import (
    adjoint_diagnostics,
    build_spacecore_operator,
    dense_reference_solve,
    make_weighted_tikhonov_problem,
    run_example,
    solve_with_spacecore,
)


def print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        widths = [max(width, len(str(value))) for width, value in zip(widths, row)]
    fmt = " | ".join(f"{{:{width}}}" for width in widths)
    print(fmt.format(*headers))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(fmt.format(*row))

## Generate a deterministic problem

The executable example owns the problem generator. We call it here rather than
duplicating construction logic in the notebook.

In this example, `G_X` and `G_Y` are diagonal SPD matrices. That keeps the implementation
on SpaceCore's public `WeightedInnerProduct` API while still making the geometry
non-Euclidean and the coordinate transpose wrong as an adjoint.

In [2]:
problem = make_weighted_tikhonov_problem(n=32, m=48, lam=1e-2, seed=0)

x_weights = np.diag(problem.Gx)
y_weights = np.diag(problem.Gy)

print(f"M.shape  = {problem.M.shape}")
print(f"Gx.shape = {problem.Gx.shape}")
print(f"Gy.shape = {problem.Gy.shape}")
print(f"lambda   = {problem.lam}")
print(f"domain metric weights nonconstant:   {np.ptp(x_weights) > 0}")
print(f"codomain metric weights nonconstant: {np.ptp(y_weights) > 0}")

assert problem.M.shape == (48, 32)
assert problem.Gx.shape == (32, 32)
assert problem.Gy.shape == (48, 48)
assert np.all(x_weights > 0)
assert np.all(y_weights > 0)
assert np.ptp(x_weights) > 0
assert np.ptp(y_weights) > 0

M.shape  = (48, 32)
Gx.shape = (32, 32)
Gy.shape = (48, 48)
lambda   = 0.01
domain metric weights nonconstant:   True
codomain metric weights nonconstant: True


## Dense NumPy reference solve

The dense solve is independent of SpaceCore and acts as the correctness oracle.
The linear system is

```python
lhs = problem.M.T @ problem.Gy @ problem.M + problem.lam * problem.Gx
rhs = problem.M.T @ problem.Gy @ problem.b
```

The helper `dense_reference_solve` performs this solve and returns diagnostics.

In [3]:
lhs = problem.M.T @ problem.Gy @ problem.M + problem.lam * problem.Gx
rhs = problem.M.T @ problem.Gy @ problem.b
x_ref, ref_diag = dense_reference_solve(problem)

print(f"lhs.shape = {lhs.shape}")
print(f"rhs.shape = {rhs.shape}")
print(f"objective value            = {ref_diag.objective:.12e}")
print(f"residual norm in Y         = {ref_diag.residual_norm_y:.12e}")
print(f"regularization norm in X   = {ref_diag.regularization_norm_x:.12e}")
print(f"first-order residual norm  = {ref_diag.first_order_residual_norm:.12e}")

assert ref_diag.first_order_residual_norm <= 1e-10

lhs.shape = (32, 32)
rhs.shape = (32,)
objective value            = 2.540443144470e-01
residual norm in Y         = 2.345650154828e-01
regularization norm in X   = 6.731031736706e+00
first-order residual norm  = 4.368925095814e-15


## SpaceCore solve

The SpaceCore path constructs the mathematical objects directly:

- `X` with the `G_X`-induced weighted inner product;
- `Y` with the `G_Y`-induced weighted inner product;
- `A : X -> Y` from the coordinate matrix `M`;
- the normal operator `A.H @ A + lambda * I_X`;
- a conjugate-gradient solve with `spacecore.cg`.

The important operation is `A.H`: it uses the metric adjoint determined by the
spaces attached to `A`.

In [4]:
ctx, X, Y, A = build_spacecore_operator(problem)

normal = A.H @ A + problem.lam * sc.IdentityLinOp(X)
rhs_sc = A.H.apply(ctx.asarray(problem.b))
result = sc.cg(
    normal,
    rhs_sc,
    tol=1e-12,
    atol=0.0,
    maxiter=2 * problem.M.shape[1],
    check_every=1,
)

sc_solve = solve_with_spacecore(problem)

print(f"X.shape = {X.shape}")
print(f"Y.shape = {Y.shape}")
print(f"A maps X -> Y: {A.domain == X and A.codomain == Y}")
print(f"CG converged: {bool(result.converged)}")
print(f"CG iterations: {int(result.num_iters)}")
print(f"CG residual norm: {float(result.residual_norm):.12e}")

np.testing.assert_allclose(np.asarray(result.x), sc_solve.x, rtol=1e-12, atol=1e-12)
assert bool(result.converged)
assert sc_solve.cg_converged

X.shape = (32,)
Y.shape = (48,)
A maps X -> Y: True
CG converged: True
CG iterations: 37
CG residual norm: 3.619974589476e-13


## Compare reference and SpaceCore solutions

The relative solution error should be near machine precision for the default NumPy run.
Small variation across BLAS implementations is acceptable; the validation tolerance below
is deliberately looser than the observed default value.

In [5]:
relative_solution_error = np.linalg.norm(sc_solve.x - x_ref) / max(1.0, np.linalg.norm(x_ref))
objective_difference = abs(sc_solve.diagnostics.objective - ref_diag.objective)

print_table(
    ["quantity", "reference", "SpaceCore", "difference"],
    [
        [
            "objective value",
            f"{ref_diag.objective:.6e}",
            f"{sc_solve.diagnostics.objective:.6e}",
            f"{objective_difference:.6e}",
        ],
        ["relative solution error", "--", "--", f"{relative_solution_error:.6e}"],
        [
            "first-order residual norm",
            f"{ref_diag.first_order_residual_norm:.6e}",
            f"{sc_solve.diagnostics.first_order_residual_norm:.6e}",
            "--",
        ],
        ["CG converged", "--", str(sc_solve.cg_converged), "--"],
        ["CG iterations", "--", str(sc_solve.cg_num_iters), "--"],
    ],
)

assert relative_solution_error <= 1e-8
assert objective_difference <= 1e-8
assert sc_solve.diagnostics.first_order_residual_norm <= 1e-8

quantity                  | reference    | SpaceCore    | difference  
--------------------------+--------------+--------------+-------------
objective value           | 2.540443e-01 | 2.540443e-01 | 0.000000e+00
relative solution error   | --           | --           | 1.480233e-14
first-order residual norm | 4.368925e-15 | 3.981618e-13 | --          
CG converged              | --           | True         | --          
CG iterations             | --           | 37           | --          


## Demonstrate the adjoint identity

The correct metric adjoint satisfies

$$
\langle Ax, y\rangle_Y = \langle x, A^\sharp y\rangle_X.
$$

The wrong coordinate-transpose test asks whether

$$
\langle Ax, y\rangle_Y \stackrel{?}{=} \langle x, M^T y\rangle_X.
$$

For non-Euclidean weighted spaces, the first error should be near floating-point
precision and the second should be visibly nonzero.

In [6]:
adj = adjoint_diagnostics(problem)

print(f"metric-adjoint identity error  = {adj.metric_identity_error:.12e}")
print(f"wrong-transpose identity error = {adj.wrong_transpose_identity_error:.12e}")

assert adj.metric_identity_error <= 1e-10
assert adj.wrong_transpose_identity_error >= 1e-3

metric-adjoint identity error  = 0.000000000000e+00
wrong-transpose identity error = 1.419221618497e+01


## Summary

- The same coordinate matrix `M` has different adjoints depending on the spaces.
- The dense reference system contains both `G_X` and `G_Y`.
- SpaceCore keeps these metrics attached to `X` and `Y`.
- `A.H` is therefore the mathematically correct metric adjoint `A^sharp`.
- This prevents a silent but serious bug common in weighted inverse problems.

In [7]:
summary = run_example()
print(f"relative solution error        = {summary['relative_solution_error']:.12e}")
print(f"objective difference           = {summary['objective_difference']:.12e}")
print(f"metric-adjoint identity error  = {summary['adjoint'].metric_identity_error:.12e}")
print(f"wrong-transpose identity error = {summary['adjoint'].wrong_transpose_identity_error:.12e}")

assert summary["relative_solution_error"] <= 1e-8
assert summary["objective_difference"] <= 1e-8
assert summary["adjoint"].metric_identity_error <= 1e-10
assert summary["adjoint"].wrong_transpose_identity_error >= 1e-3

relative solution error        = 1.480232989290e-14
objective difference           = 0.000000000000e+00
metric-adjoint identity error  = 0.000000000000e+00
wrong-transpose identity error = 1.419221618497e+01
